# SparkXGB Benchmark (Parameterized)

Train XGBoost using SparkXGBClassifier (distributed CPU).

**Key decisions for large scale:**
- Drop StringType categoricals — VectorAssembler on wide tables causes OOM.
- Use `persist(DISK_ONLY)` instead of `.cache()` — spills to disk instead of forcing everything into memory.
- No unnecessary `.count()` calls — each triggers a full pass.
- Use `features_col` (singular) with VectorAssembler — `features_cols` (plural, no assembler) requires `device=cuda` (GPU only).

**Parameters (via widgets):**
- `data_size`: tiny/small/medium/medium_large/large/xlarge
- `num_workers`: Number of XGBoost workers (0=auto, uses cluster executor count)
- `node_type`: Node type label for run naming

In [ ]:
import time
import sys, os
import mlflow

# --- Widget parameters (overridable via job base_parameters) ---
dbutils.widgets.dropdown("data_size", "large", ["tiny", "small", "medium", "medium_large", "large", "xlarge"], "Data Size")
dbutils.widgets.text("num_workers", "0", "Num Workers (0=auto)")
dbutils.widgets.text("node_type", "D16sv5", "Node Type")
dbutils.widgets.text("catalog", "brian_gen_ai", "Catalog")
dbutils.widgets.text("schema", "xgb_scaling", "Schema")
dbutils.widgets.text("table_name", "", "Table Name (override)")

data_size = dbutils.widgets.get("data_size")
num_workers_input = int(dbutils.widgets.get("num_workers"))
node_type = dbutils.widgets.get("node_type")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table_name_override = dbutils.widgets.get("table_name").strip()

# Resolve table from preset
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = "/".join(notebook_path.split("/")[:-2])
sys.path.insert(0, f"/Workspace{repo_root}")
from src.config import get_preset

preset = get_preset(data_size)

if table_name_override:
    TABLE = f"{catalog}.{schema}.{table_name_override}"
else:
    TABLE = f"{catalog}.{schema}.imbalanced_{preset.table_suffix}"
LABEL_COL = "label"

# Auto-detect executor count for num_workers=0
if num_workers_input <= 0:
    sc = spark.sparkContext
    num_xgb_workers = max(1, sc.defaultParallelism // int(spark.conf.get("spark.task.cpus", "1")))
else:
    num_xgb_workers = num_workers_input

# Get experiment path from current user
user_email = spark.sql("SELECT current_user()").collect()[0][0]
EXPERIMENT_PATH = f"/Users/{user_email}/xgb_scaling_benchmark"

# Build run name — use table_name_override if set, otherwise preset suffix
table_label = table_name_override if table_name_override else f"{preset.table_suffix}_sparkxgb_v2"
RUN_NAME = f"{table_label}_sparkxgb_{num_xgb_workers}w_{node_type.lower()}"

t0 = time.time()
print(f"Table: {TABLE}")
print(f"Data size: {data_size} ({preset.rows:,} rows, {preset.total_features} features)")
print(f"XGBoost workers: {num_xgb_workers}")
print(f"Run name: {RUN_NAME}")

## Load Data & Identify Features

In [ ]:
t_load = time.time()
df = spark.read.table(TABLE)

schema = df.schema
numeric_features = [
    f.name for f in schema.fields
    if f.name != LABEL_COL
    and str(f.dataType) in ("DoubleType()", "FloatType()", "IntegerType()", "LongType()")
]
cat_features = [
    f.name for f in schema.fields
    if f.name != LABEL_COL and str(f.dataType) == "StringType()"
]

# Drop categoricals — VectorAssembler on 100M × 500 causes OOM
feature_cols = numeric_features
n_features = len(feature_cols)
print(f"Using {n_features} numeric features (dropping {len(cat_features)} StringType categoricals)")

# Select only needed columns to reduce shuffle/storage
df = df.select(feature_cols + [LABEL_COL])
load_seconds = time.time() - t_load
print(f"Schema loaded in {load_seconds:.1f}s")

## VectorAssembler + Split

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark import StorageLevel

t_feat = time.time()
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="keep")
df_assembled = assembler.transform(df).select("features", LABEL_COL)
# DO NOT .cache() — 100M × 400 dense vectors = ~320 GB, exceeds cluster memory.
# VectorAssembler is lazy; Spark handles spill.

train_df, test_df = df_assembled.randomSplit([0.8, 0.2], seed=42)
# DISK_ONLY avoids OOM while preventing recomputation during training
train_df.persist(StorageLevel.DISK_ONLY)
test_df.persist(StorageLevel.DISK_ONLY)
feat_split_seconds = time.time() - t_feat
print(f"Assemble + split (lazy) in {feat_split_seconds:.1f}s")

## Scale Pos Weight

In [ ]:
# Get class balance from metadata (avoids extra pass over 100M rows)
label_counts = spark.sql(f"SELECT label, COUNT(*) as cnt FROM {TABLE} GROUP BY label").collect()
counts = {int(r["label"]): r["cnt"] for r in label_counts}
neg = counts.get(0, 1)
pos = counts.get(1, 1)
row_count = neg + pos
scale_pos_weight = round(neg / pos, 2)
print(f"Rows: {row_count:,}, scale_pos_weight: {scale_pos_weight}")

## Train SparkXGBClassifier

In [ ]:
from xgboost.spark import SparkXGBClassifier

t_train = time.time()
xgb = SparkXGBClassifier(
    features_col="features",
    label_col=LABEL_COL,
    num_workers=num_xgb_workers,
    use_gpu=False,
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    eval_metric="logloss",
)
xgb_model = xgb.fit(train_df)
train_seconds = time.time() - t_train
print(f"Training done in {train_seconds:.1f}s (num_workers={num_xgb_workers})")

## Evaluate

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

t_eval = time.time()
predictions = xgb_model.transform(test_df)

auc_roc = BinaryClassificationEvaluator(labelCol=LABEL_COL, metricName="areaUnderROC").evaluate(predictions)
auc_pr = BinaryClassificationEvaluator(labelCol=LABEL_COL, metricName="areaUnderPR").evaluate(predictions)
f1 = MulticlassClassificationEvaluator(labelCol=LABEL_COL, metricName="f1").evaluate(predictions)
eval_seconds = time.time() - t_eval

total_seconds = time.time() - t0
print(f"AUC-ROC: {auc_roc:.6f}")
print(f"AUC-PR:  {auc_pr:.6f}")
print(f"F1:      {f1:.6f}")
print(f"Load: {load_seconds:.1f}s | Feat+Split: {feat_split_seconds:.1f}s | Train: {train_seconds:.1f}s | Eval: {eval_seconds:.1f}s | Total: {total_seconds:.1f}s")

## Log to MLflow

In [ ]:
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params({
        "approach": "sparkxgb",
        "input_table": TABLE,
        "data_size": data_size,
        "n_rows": str(row_count),
        "n_features": str(n_features),
        "num_workers": str(num_xgb_workers),
        "node_type": node_type,
        "trainer": "SparkXGBClassifier",
        "max_depth": "8",
        "n_estimators": "200",
        "scale_pos_weight": str(scale_pos_weight),
        "storage_level": "DISK_ONLY",
    })
    mlflow.log_metrics({
        "auc_roc": auc_roc,
        "auc_pr": auc_pr,
        "f1": f1,
        "data_load_seconds": load_seconds,
        "feat_split_seconds": feat_split_seconds,
        "train_time_seconds": train_seconds,
        "eval_seconds": eval_seconds,
        "total_time_seconds": total_seconds,
    })

print("MLflow logged. Done.")

In [ ]:
import json

result = {
    "status": "ok",
    "run_name": RUN_NAME,
    "training_mode": "spark_xgboost_v2",
    "data_size": data_size,
    "node_type": node_type,
    "n_rows": row_count,
    "n_features": n_features,
    "num_workers": num_xgb_workers,
    "auc_roc": round(auc_roc, 4),
    "auc_pr": round(auc_pr, 4),
    "f1": round(f1, 4),
    "train_time_sec": round(train_seconds, 1),
    "total_time_sec": round(total_seconds, 1),
}
result_json = json.dumps(result)
print(f"\nNotebook result: {result_json}")
dbutils.notebook.exit(result_json)